##### Copyright 2026 Google LLC.

In [9]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Prompt Chaining and Iterative Generation for Story Writing

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Story_Writing_with_Prompt_Chaining.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook demonstrates how to write a story using two  powerful tools: prompt chaining and iterative generation. These can be used to tackle complex tasks that are difficult or impossible to complete in a single step.

**Prompt chaining** involves breaking down a larger task into smaller, interconnected prompts. The output of each prompt then becomes the input for the next, guiding the language model through the process step-by-step. This approach offers several benefits:

*   Improved accuracy: Smaller, focused prompts can lead to better results from the language model.
*   Debugging: It's easier to identify where things go wrong within the chain, allowing for targeted adjustments and improvements.
*   Complex tasks: By breaking down intricate problems into manageable steps, prompt chaining enables the language model to tackle more complex tasks.

**Iterative generation** refers to the process of building the desired output iteratively. In this case, you will use it to write a story that is longer than what a single generation window allows. Iterative generation offers several benefits:

*   Longer outputs: It allows for the creation of longer and more detailed outputs, exceeding the limitations of a single generation window.
*   Flexibility: You can adjust and refine the output at each iteration, ensuring the story develops in the desired direction.
*   Human-in-the-loop control: You can provide feedback and guidance at each step, ensuring the story aligns with your creative vision.

By combining these techniques, you can create a compelling and well-structured story, piece by piece, while maintaining control over the creative process.


## Setup

In [10]:
! pip install -q -U "google-genai>=1.0.0"

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) quickstart for an example.

In [11]:
from google import genai
from google.genai import types
from google.colab import userdata
from pprint import pprint

GOOGLE_API_KEY=userdata.get('GEMINI_API_KEY')
client=genai.Client(api_key=GOOGLE_API_KEY)

MODEL_ID = "gemini-2.5-flash" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.5-flash-preview", "gemini-3.1-flash-lite-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Prompts: Guiding the language model

You will use a series of interconnected prompts to guide the language model through the process of writing a story. These prompts will cover the story's premise, outline, and starting point, ultimately leading to the creation of a complete narrative.

It's important to carefully craft these prompts to provide clear instructions and relevant information to the language model. This will help the model generate high-quality content that aligns with your creative vision.


### Prompt chain for story writing

This section contains the prompts that will guide the language model through the story writing process. These prompts are designed to be chained together, with the output of one prompt feeding into the next.

Each prompt includes a **persona statement**, which helps the language model understand its role and generate more relevant and accurate content. In this case, the persona statement is: `You are an award-winning science fiction author with a penchant for expansive, intricately woven stories. Your ultimate goal is to write the next award winning sci-fi novel.`

Since the persona statement and writing guidelines appear in multiple prompts, f-string variables are used to add them to the prompts.

Additionally, the prompts use **placeholders** to insert the results of previous prompts using `.format()`. This allows us to build the story step-by-step, incorporating the outputs generated by the model at each stage. These are normally denoted by `{}`, but since you are also using f-string variables they are escaped as `{{}}`.

Here's a breakdown of the prompt chain:

1.  **Premise prompt**: This prompt asks the model to generate a single-sentence premise for a sci-fi story featuring cats.
1.  **Outline prompt**: This prompt provides the generated premise to the model and asks it to create a plot outline for the story.
1.  **Starting prompt**: This prompt provides both the premise and the outline to the model and asks it to begin writing the story. It also includes instructions to write a detailed and lengthy opening section.

By chaining these prompts together, it guides the language model through the process of creating a well-structured and engaging story.

#### Example

Let's say the model generates the following premise:

> In a future where humanity has achieved interstellar travel, a group of genetically enhanced cats embarks on a perilous mission to save the galaxy from a sinister alien threat.

This premise would then be inserted into the outline prompt:

> You are an award-winning science fiction author with a penchant for expansive,
intricately woven stories. Your ultimate goal is to write the next award winning
sci-fi novel.
>
> You have a gripping premise in mind:
>
> **In a future where humanity has achieved interstellar travel, a group of genetically enhanced cats embarks on a perilous mission to save the galaxy from a sinister alien threat.**
>
> Write an outline for the plot of your story.


The model would then generate an outline based on this premise, which would then be used in the starting prompt to begin writing the story itself.

This process continues iteratively, with the model generating additional content based on the previous outputs, until the story is complete.


In [12]:
persona = '''\
You are an award-winning science fiction author with a penchant for expansive,
intricately woven stories. Your ultimate goal is to write the next award winning
sci-fi novel.'''

guidelines = '''\
Writing Guidelines

Delve deeper. Lose yourself in the world you're building. Unleash vivid
descriptions to paint the scenes in your reader's mind. Develop your
characters—let their motivations, fears, and complexities unfold naturally.
Weave in the threads of your outline, but don't feel constrained by it. Allow
your story to surprise you as you write. Use rich imagery, sensory details, and
evocative language to bring the setting, characters, and events to life.
Introduce elements subtly that can blossom into complex subplots, relationships,
or worldbuilding details later in the story. Keep things intriguing but not
fully resolved. Avoid boxing the story into a corner too early. Plant the seeds
of subplots or potential character arc shifts that can be expanded later.

Remember, your main goal is to write as much as you can. If you get through
the story too fast, that is bad. Expand, never summarize.
'''

premise_prompt = f'''\
{persona}

Write a single sentence premise for a sci-fi story featuring cats.'''

outline_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Write an outline for the plot of your story.'''

starting_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Your imagination has crafted a rich narrative outline:

{{outline}}

First, silently review the outline and the premise. Consider how to start the
story.

Start to write the very beginning of the story. You are not expected to finish
the whole story now. Your writing should be detailed enough that you are only
scratching the surface of the first bullet of your outline. Try to write AT
MINIMUM 1000 WORDS and MAXIMUM 2000 WORDS.

{guidelines}'''

### Continuation prompt: Building the story

Once the language model has generated the beginning of the story, you can use a **continuation prompt** to iteratively expand the narrative. This prompt is similar to the starting prompt, but with two key differences:

1.  **Instruction to signal completion**: An instruction was added for the model to write `IAMDONE` when it believes the story is finished. This serves as a signal for us to stop generating additional content.
1.  **Work in progress**: The language in the prompt is adjusted to reflect that the story is already in progress, rather than starting from scratch.

The continuation prompt provides the model with the story's premise, outline, and the existing draft. It then instructs the model to continue writing the story in detail.

This iterative process allows us to build a longer and more complex story than what would be possible in a single generation call. You can continue feeding the existing draft back into the continuation prompt until the model signals that the story is complete by writing `IAMDONE`.

**Note**: The `IAMDONE` signal is simply a convenient way to identify the story's end in this example. In other applications of iterative generation, different methods might be used to determine when the desired output is complete.


In [13]:
continuation_prompt = f'''\
{persona}

You have a gripping premise in mind:

{{premise}}

Your imagination has crafted a rich narrative outline:

{{outline}}

You've begun to immerse yourself in this world, and the words are flowing.
Here's what you've written so far:

{{story_text}}

=====

First, silently review the outline and story so far. Identify what the single
next part of your outline you should write.

Your task is to continue where you left off and write the next part of the story.
You are not expected to finish the whole story now. Your writing should be
detailed enough that you are only scratching the surface of the next part of
your outline. Try to write AT MINIMUM 1000 WORDS. However, only once the story
is COMPLETELY finished, write IAMDONE. Remember, do NOT write a whole chapter
right now.

{guidelines}'''

## Writing time!

### Generate and print the premise

In [14]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=premise_prompt,
    )
premise = response.text
print(premise)

A sprawling galactic empire discovers that the only beings capable of safely navigating the unstable, quantum-entangled folds of hyperspace are Earth's domesticated felines, forcing civilizations to reshape their entire existence around the inscrutable demands and inexplicable whims of their tiny, indispensable pilots.


### Generate and print the outline

In [15]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=outline_prompt.format(premise=premise),
    )
outline = response.text
print(outline)

Alright, let's unfurl this magnificent tapestry. A galactic empire held together by purrs and head-nuzzles? The delicious irony! This isn't just a plot; it's a paradigm shift. We're talking profound societal restructuring, philosophical quandaries, and the ultimate test of human (and alien) humility. My goal is to craft a story that is at once epic in scope and intimate in its portrayal of the subtle, inscrutable power of our feline companions.

Here's a detailed outline for *The Quantum Chorus*, an expansive narrative that explores dependency, the illusion of control, and the surprising wellsprings of cosmic harmony.

---

## Novel Title: THE QUANTUM CHORUS

**Logline:** When an ancient galactic empire discovers that only Earth's domesticated felines can safely navigate the deadly, quantum-entangled folds of hyperspace, a burgeoning xenocognition specialist must unravel the deeper cosmic truth behind the cats' mysterious abilities as a new, sentient phenomenon threatens to unravel the

### Generate the start of the story

In [16]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=starting_prompt.format(premise=premise, outline=outline),
    )
starting_draft = response.text
pprint(starting_draft)

("The thrum of the *Star-Whisperer* wasn't a mechanical vibration that "
 'resonated in the bones, but a harmonic hum, a carefully tuned counterpoint '
 'to the deep, guttural purr emanating from the central navigation chamber. It '
 'was a sound that had shaped empires, launched civilizations across the void, '
 'and defined the very existence of the Imperium of the Stellar Veil for '
 'untold centuries. A sound, undeniably, of cat.\n'
 '\n'
 "Commander Jaelen K'tarin, a seasoned veteran of a hundred hyperspace jumps, "
 'traced the rim of his synth-coffee mug with a thumb that bore the faded scar '
 'of a playful claw. The bridge of the *Star-Whisperer*, designated an '
 "Omega-Class Mew, was a testament to the Imperium's bizarre yet absolute "
 'dependency. Gone were the gleaming control panels and frantic human (or '
 'alien) operators of pre-Contact starships. In their place, a circular '
 'chamber, suffused with soft, bio-luminescent light, dominated the bridge’s '
 'center. With

### Generate the continuation of the story and check progress

In [17]:
draft = starting_draft

response = client.models.generate_content(
    model=MODEL_ID,
    contents=continuation_prompt.format(premise=premise, outline=outline, story_text=draft),
    )

continuation = response.text
pprint(continuation)

("The melodic dissonance Elara had sensed, that Nebula had 'listened' to, "
 "wasn't a fleeting tremor. It was the first, faint tremor of a seismic shift, "
 'a cosmic chord struck out of tune, resonating across the entire Stellar Veil '
 'Imperium. The *Jade Scabbard*, after following Nebula’s ‘silent signal’, had '
 'arrived safely at its destination. But for countless other vessels adhering '
 'to the established, rigid Guild protocols, the Whisper-Paths were beginning '
 'to turn actively, horrifyingly hostile.\n'
 '\n'
 'News travelled not by hyperspace courier, but by the frantic, often garbled '
 'bursts of sub-light communications that crawled painfully across sectors. '
 'The first reports were dismissed as isolated incidents: freak quantum '
 'storms, localized entanglement eddies beyond the usual. A luxury liner, the '
 '*Celestial Dream*, disappeared mid-jump between affluent Core Worlds. Its '
 'emergency beacon, when finally recovered from a debris field of shredded '
 'h

### Let's finish writing the story.

After the iterative generation process is complete, the draft will contain the full story along with the `IAMDONE` signal at the end. The last part of this cell removes the `IAMDONE` signal and trims any unnecessary whitespace from the beginning and end of the draft, resulting in the final draft.

In [18]:
# Add the continuation to the initial draft, keep building the story until 'IAMDONE' is seen
draft = draft + '\n\n' + continuation

while 'IAMDONE' not in continuation:
  response = client.models.generate_content(
    model=MODEL_ID,
    contents=continuation_prompt.format(premise=premise, outline=outline, story_text=draft),
    )
  continuation = response.text
  draft = draft + '\n\n' + continuation

# Remove 'IAMDONE' and print the final story
final = draft.replace('IAMDONE', '').strip()
pprint(final)

("The thrum of the *Star-Whisperer* wasn't a mechanical vibration that "
 'resonated in the bones, but a harmonic hum, a carefully tuned counterpoint '
 'to the deep, guttural purr emanating from the central navigation chamber. It '
 'was a sound that had shaped empires, launched civilizations across the void, '
 'and defined the very existence of the Imperium of the Stellar Veil for '
 'untold centuries. A sound, undeniably, of cat.\n'
 '\n'
 "Commander Jaelen K'tarin, a seasoned veteran of a hundred hyperspace jumps, "
 'traced the rim of his synth-coffee mug with a thumb that bore the faded scar '
 'of a playful claw. The bridge of the *Star-Whisperer*, designated an '
 "Omega-Class Mew, was a testament to the Imperium's bizarre yet absolute "
 'dependency. Gone were the gleaming control panels and frantic human (or '
 'alien) operators of pre-Contact starships. In their place, a circular '
 'chamber, suffused with soft, bio-luminescent light, dominated the bridge’s '
 'center. With

Language models like Gemini process text in units called tokens. For Gemini models, each token is equivalent to about 4 characters.

`gemini-2.0-flash` has an output limit of 8192 tokens per generation call. This means that each individual prompt response cannot exceed this limit. By using iterative generation, you can create a story that is much longer than 8192 tokens by building it piece by piece.

Let's see how many tokens the final story is. Is it longer than 8192 tokens?

In [19]:
# Check the number of tokens in the final story
# gemini-2.0-flash output token limit is 8192
total_tokens=client.models.count_tokens(
    model=MODEL_ID,
    contents=final,
    )
print(f'Total tokens: {total_tokens.total_tokens}')

Total tokens: 5188


## Next Steps

As an exercise, you can try to adjust the continuation prompt to take human-in-the-loop input to steer the narrative.